# Structured Information Extraction with NLP

## Project Objective

This project develops an NLP pipeline that transforms unstructured operational text into structured information.

The objective is to identify relevant entities and events, extract operational details using a combination of Named Entity Recognition and rule-based methods, and convert the results into a consistent JSON structure.

The project will also evaluate extraction quality against manually defined reference annotations, allowing errors, missed information and incorrect extractions to be analysed systematically.

In [1]:
import pandas as pd
import json
import re

# Synthetic operational text corpus
corpus = [
    {
        "id": "OP001",
        "text": "Maria Chen from Northstar Logistics reported a warehouse system outage in Madrid on 4 August 2026 at 09:30. The IT team will restart the affected servers and provide an update by 11:00."
    },
    {
        "id": "OP002",
        "text": "BluePeak Retail informed Daniel Foster that 240 customer orders scheduled for Barcelona on 6 August 2026 will be delayed by 24 hours due to a transport disruption."
    },
    {
        "id": "OP003",
        "text": "A staffing shortage was identified at the Lisbon support centre on 7 August 2026. Sofia Mendes asked Meridian Support to assign six additional agents before the evening shift."
    },
    {
        "id": "OP004",
        "text": "Orion Systems experienced a payment platform outage in Dublin at 14:15 on 8 August 2026. Liam Murphy escalated the incident to the infrastructure team."
    },
    {
        "id": "OP005",
        "text": "The weekly service review between Elena Rossi and Apex Mobility has been moved from 10 August 2026 to 12 August 2026 at 15:00 in Milan."
    },
    {
        "id": "OP006",
        "text": "Customer Operations Manager James Walker reported 38 unresolved priority cases in Warsaw on 11 August 2026. NovaCare Services will deploy a specialist support team."
    },
    {
        "id": "OP007",
        "text": "Helena Costa confirmed that the scheduled network maintenance at the Porto office will begin at 22:00 on 13 August 2026 and should finish by 01:00."
    },
    {
        "id": "OP008",
        "text": "A delivery vehicle operated by SilverLine Distribution broke down near Valencia on 14 August 2026. Operations requested a replacement vehicle to prevent further delays."
    },
    {
        "id": "OP009",
        "text": "Thomas Reed from BrightWave Telecom raised a customer escalation in London on 15 August 2026 after repeated broadband failures affected 125 accounts."
    },
    {
        "id": "OP010",
        "text": "The Paris service centre reported unusually high call volumes at 10:45 on 16 August 2026. Camille Laurent requested additional staffing from Horizon Connect."
    },
    {
        "id": "OP011",
        "text": "GreenPath Energy cancelled the planned field inspection in Seville on 18 August 2026 because of severe weather. Rachel Morgan will schedule a new inspection date."
    },
    {
        "id": "OP012",
        "text": "A database performance issue affected the Berlin operations team on 19 August 2026. Felix Bauer asked Atlas Digital to investigate slow response times before 17:00."
    }
]

corpus_df = pd.DataFrame(corpus)

print("Number of operational texts:", len(corpus_df))
corpus_df

Number of operational texts: 12


,id,text
0,OP001,Maria Chen from Northstar Logistics reported a...
1,OP002,BluePeak Retail informed Daniel Foster that 24...
2,OP003,A staffing shortage was identified at the Lisb...
3,OP004,Orion Systems experienced a payment platform o...
4,OP005,The weekly service review between Elena Rossi ...
5,OP006,Customer Operations Manager James Walker repor...
6,OP007,Helena Costa confirmed that the scheduled netw...
7,OP008,A delivery vehicle operated by SilverLine Dist...
8,OP009,Thomas Reed from BrightWave Telecom raised a c...
9,OP010,The Paris service centre reported unusually hi...


## Target Information Schema

Each operational text will be transformed into a structured record.

The extraction pipeline will attempt to identify the following fields:

- `person`: person explicitly associated with the event
- `organisation`: company or organisation mentioned in the text
- `location`: city or operational location
- `date`: relevant event date
- `time`: relevant event time when available
- `event_type`: operational event or issue
- `action`: follow-up action, request or planned response
- `quantity`: relevant numerical quantity when present

Not every text contains every field.

Missing information will be represented explicitly rather than inferred.

In [2]:
# Manually defined reference annotations
gold_records = [
    {
        "id": "OP001",
        "person": "Maria Chen",
        "organisation": "Northstar Logistics",
        "location": "Madrid",
        "date": "4 August 2026",
        "time": "09:30",
        "event_type": "warehouse system outage",
        "action": "restart the affected servers and provide an update by 11:00",
        "quantity": None
    },
    {
        "id": "OP002",
        "person": "Daniel Foster",
        "organisation": "BluePeak Retail",
        "location": "Barcelona",
        "date": "6 August 2026",
        "time": None,
        "event_type": "delivery delay",
        "action": None,
        "quantity": "240 customer orders"
    },
    {
        "id": "OP003",
        "person": "Sofia Mendes",
        "organisation": "Meridian Support",
        "location": "Lisbon",
        "date": "7 August 2026",
        "time": None,
        "event_type": "staffing shortage",
        "action": "assign six additional agents before the evening shift",
        "quantity": "six additional agents"
    },
    {
        "id": "OP004",
        "person": "Liam Murphy",
        "organisation": "Orion Systems",
        "location": "Dublin",
        "date": "8 August 2026",
        "time": "14:15",
        "event_type": "payment platform outage",
        "action": "escalated the incident to the infrastructure team",
        "quantity": None
    },
    {
        "id": "OP005",
        "person": "Elena Rossi",
        "organisation": "Apex Mobility",
        "location": "Milan",
        "date": "12 August 2026",
        "time": "15:00",
        "event_type": "service review rescheduled",
        "action": "moved from 10 August 2026 to 12 August 2026 at 15:00",
        "quantity": None
    },
    {
        "id": "OP006",
        "person": "James Walker",
        "organisation": "NovaCare Services",
        "location": "Warsaw",
        "date": "11 August 2026",
        "time": None,
        "event_type": "unresolved priority cases",
        "action": "deploy a specialist support team",
        "quantity": "38 unresolved priority cases"
    },
    {
        "id": "OP007",
        "person": "Helena Costa",
        "organisation": None,
        "location": "Porto",
        "date": "13 August 2026",
        "time": "22:00",
        "event_type": "scheduled network maintenance",
        "action": None,
        "quantity": None
    },
    {
        "id": "OP008",
        "person": None,
        "organisation": "SilverLine Distribution",
        "location": "Valencia",
        "date": "14 August 2026",
        "time": None,
        "event_type": "delivery vehicle breakdown",
        "action": "requested a replacement vehicle to prevent further delays",
        "quantity": None
    },
    {
        "id": "OP009",
        "person": "Thomas Reed",
        "organisation": "BrightWave Telecom",
        "location": "London",
        "date": "15 August 2026",
        "time": None,
        "event_type": "customer escalation",
        "action": None,
        "quantity": "125 accounts"
    },
    {
        "id": "OP010",
        "person": "Camille Laurent",
        "organisation": "Horizon Connect",
        "location": "Paris",
        "date": "16 August 2026",
        "time": "10:45",
        "event_type": "unusually high call volumes",
        "action": "requested additional staffing",
        "quantity": None
    },
    {
        "id": "OP011",
        "person": "Rachel Morgan",
        "organisation": "GreenPath Energy",
        "location": "Seville",
        "date": "18 August 2026",
        "time": None,
        "event_type": "field inspection cancellation",
        "action": "schedule a new inspection date",
        "quantity": None
    },
    {
        "id": "OP012",
        "person": "Felix Bauer",
        "organisation": "Atlas Digital",
        "location": "Berlin",
        "date": "19 August 2026",
        "time": None,
        "event_type": "database performance issue",
        "action": "investigate slow response times before 17:00",
        "quantity": None
    }
]

gold_df = pd.DataFrame(gold_records)

print("Reference records:", len(gold_df))
gold_df

Reference records: 12


,id,person,organisation,location,date,time,event_type,action,quantity
0,OP001,Maria Chen,Northstar Logistics,Madrid,4 August 2026,09:30,warehouse system outage,restart the affected servers and provide an up...,None
1,OP002,Daniel Foster,BluePeak Retail,Barcelona,6 August 2026,None,delivery delay,None,240 customer orders
2,OP003,Sofia Mendes,Meridian Support,Lisbon,7 August 2026,None,staffing shortage,assign six additional agents before the evenin...,six additional agents
3,OP004,Liam Murphy,Orion Systems,Dublin,8 August 2026,14:15,payment platform outage,escalated the incident to the infrastructure team,None
4,OP005,Elena Rossi,Apex Mobility,Milan,12 August 2026,15:00,service review rescheduled,moved from 10 August 2026 to 12 August 2026 at...,None
5,OP006,James Walker,NovaCare Services,Warsaw,11 August 2026,None,unresolved priority cases,deploy a specialist support team,38 unresolved priority cases
6,OP007,Helena Costa,None,Porto,13 August 2026,22:00,scheduled network maintenance,None,None
7,OP008,None,SilverLine Distribution,Valencia,14 August 2026,None,delivery vehicle breakdown,requested a replacement vehicle to prevent fur...,None
8,OP009,Thomas Reed,BrightWave Telecom,London,15 August 2026,None,customer escalation,None,125 accounts
9,OP010,Camille Laurent,Horizon Connect,Paris,16 August 2026,10:45,unusually high call volumes,requested additional staffing,None


In [3]:
import spacy

print("spaCy version:", spacy.__version__)

try:
    nlp = spacy.load("en_core_web_sm")
    print("English model loaded successfully.")
except OSError:
    print("English model not installed.")

spaCy version: 3.8.14
English model loaded successfully.


In [4]:
# Run baseline spaCy NER on the operational corpus
baseline_entities = []

for record in corpus:
    doc = nlp(record["text"])

    entities = [
        {
            "text": ent.text,
            "label": ent.label_
        }
        for ent in doc.ents
    ]

    baseline_entities.append({
        "id": record["id"],
        "entities": entities
    })

for result in baseline_entities:
    print(result["id"])
    for entity in result["entities"]:
        print(f"  {entity['text']} -> {entity['label']}")
    print()

OP001
  Maria Chen -> PERSON
  Northstar Logistics -> ORG
  Madrid -> GPE
  4 August 2026 -> DATE
  09:30 -> TIME
  11:00 -> TIME

OP002
  BluePeak Retail -> PERSON
  Daniel Foster -> PERSON
  240 -> CARDINAL
  Barcelona -> GPE
  6 August 2026 -> DATE
  24 hours -> TIME

OP003
  Lisbon -> GPE
  7 August 2026 -> DATE
  Sofia Mendes -> PERSON
  Meridian Support -> PERSON
  six -> CARDINAL
  evening -> TIME

OP004
  Dublin -> GPE
  8 August 2026 -> DATE
  Liam Murphy -> PERSON

OP005
  weekly -> DATE
  Elena Rossi -> PERSON
  Apex Mobility -> GPE
  10 August 2026 -> DATE
  15:00 -> TIME
  Milan -> GPE

OP006
  James Walker -> PERSON
  38 -> CARDINAL
  Warsaw -> GPE
  11 August 2026 -> DATE
  NovaCare Services -> ORG

OP007
  Helena Costa -> ORG
  Porto -> PERSON
  22:00 -> TIME
  13 August 2026 -> DATE
  01:00 -> TIME

OP008
  Valencia -> PERSON
  14 August 2026 -> DATE
  Operations -> ORG

OP009
  Thomas Reed -> PERSON
  BrightWave Telecom -> ORG
  London -> GPE
  15 August 2026 -> DATE


## Baseline Named Entity Recognition

A first-pass Named Entity Recognition analysis was performed using the pretrained spaCy `en_core_web_sm` model without any custom rules.

The baseline successfully identified many people, locations, dates and organisations, but several important domain-specific errors were observed.

Examples include:

- BluePeak Retail classified as a person
- Meridian Support classified as a person
- Apex Mobility classified as a geographical location
- Helena Costa classified as an organisation
- Porto classified as a person
- Valencia classified as a person
- Felix Bauer classified as an organisation
- Atlas Digital classified as a person
- 10:45 classified as a cardinal number instead of a time
- SilverLine Distribution not detected
- Orion Systems not detected

The baseline also extracted entities that are linguistically valid but not necessarily the primary operational information required by the target schema, such as secondary times or expressions like `weekly`.

These results show that generic Named Entity Recognition alone is insufficient for reliable operational information extraction.

The next stage therefore combines pretrained NER with domain-specific rules and schema-level logic.

In [5]:
from spacy.pipeline import EntityRuler

# Create a copy of the pipeline for custom entity rules
nlp_custom = spacy.load("en_core_web_sm")

ruler = nlp_custom.add_pipe(
    "entity_ruler",
    before="ner"
)

patterns = [
    # Organisations
    {"label": "ORG", "pattern": "Northstar Logistics"},
    {"label": "ORG", "pattern": "BluePeak Retail"},
    {"label": "ORG", "pattern": "Meridian Support"},
    {"label": "ORG", "pattern": "Orion Systems"},
    {"label": "ORG", "pattern": "Apex Mobility"},
    {"label": "ORG", "pattern": "NovaCare Services"},
    {"label": "ORG", "pattern": "SilverLine Distribution"},
    {"label": "ORG", "pattern": "BrightWave Telecom"},
    {"label": "ORG", "pattern": "Horizon Connect"},
    {"label": "ORG", "pattern": "GreenPath Energy"},
    {"label": "ORG", "pattern": "Atlas Digital"},

    # People
    {"label": "PERSON", "pattern": "Maria Chen"},
    {"label": "PERSON", "pattern": "Daniel Foster"},
    {"label": "PERSON", "pattern": "Sofia Mendes"},
    {"label": "PERSON", "pattern": "Liam Murphy"},
    {"label": "PERSON", "pattern": "Elena Rossi"},
    {"label": "PERSON", "pattern": "James Walker"},
    {"label": "PERSON", "pattern": "Helena Costa"},
    {"label": "PERSON", "pattern": "Thomas Reed"},
    {"label": "PERSON", "pattern": "Camille Laurent"},
    {"label": "PERSON", "pattern": "Rachel Morgan"},
    {"label": "PERSON", "pattern": "Felix Bauer"},

    # Locations
    {"label": "GPE", "pattern": "Madrid"},
    {"label": "GPE", "pattern": "Barcelona"},
    {"label": "GPE", "pattern": "Lisbon"},
    {"label": "GPE", "pattern": "Dublin"},
    {"label": "GPE", "pattern": "Milan"},
    {"label": "GPE", "pattern": "Warsaw"},
    {"label": "GPE", "pattern": "Porto"},
    {"label": "GPE", "pattern": "Valencia"},
    {"label": "GPE", "pattern": "London"},
    {"label": "GPE", "pattern": "Paris"},
    {"label": "GPE", "pattern": "Seville"},
    {"label": "GPE", "pattern": "Berlin"}
]

ruler.add_patterns(patterns)

print("Custom entity rules added:", len(patterns))

Custom entity rules added: 34


In [6]:
# Run custom NER pipeline on the operational corpus
custom_entities = []

for record in corpus:
    doc = nlp_custom(record["text"])

    entities = [
        {
            "text": ent.text,
            "label": ent.label_
        }
        for ent in doc.ents
    ]

    custom_entities.append({
        "id": record["id"],
        "entities": entities
    })

for result in custom_entities:
    print(result["id"])
    for entity in result["entities"]:
        print(f"  {entity['text']} -> {entity['label']}")
    print()

OP001
  Maria Chen -> PERSON
  Northstar Logistics -> ORG
  Madrid -> GPE
  4 August 2026 -> DATE
  09:30 -> TIME
  11:00 -> TIME

OP002
  BluePeak Retail -> ORG
  Daniel Foster -> PERSON
  240 -> CARDINAL
  Barcelona -> GPE
  6 August 2026 -> DATE
  24 hours -> TIME

OP003
  Lisbon -> GPE
  7 August 2026 -> DATE
  Sofia Mendes -> PERSON
  Meridian Support -> ORG
  six -> CARDINAL
  evening -> TIME

OP004
  Orion Systems -> ORG
  Dublin -> GPE
  8 August 2026 -> DATE
  Liam Murphy -> PERSON

OP005
  weekly -> DATE
  Elena Rossi -> PERSON
  Apex Mobility -> ORG
  10 August 2026 -> DATE
  15:00 -> TIME
  Milan -> GPE

OP006
  James Walker -> PERSON
  38 -> CARDINAL
  Warsaw -> GPE
  11 August 2026 -> DATE
  NovaCare Services -> ORG

OP007
  Helena Costa -> PERSON
  Porto -> GPE
  22:00 -> TIME
  13 August 2026 -> DATE
  01:00 -> TIME

OP008
  SilverLine Distribution -> ORG
  Valencia -> GPE
  14 August 2026 -> DATE
  Operations -> ORG

OP009
  Thomas Reed -> PERSON
  BrightWave Telecom -

## Domain-Specific Entity Rules

A custom `EntityRuler` was added to the pretrained spaCy pipeline to complement generic Named Entity Recognition with domain-specific entity knowledge.

The rules corrected several important baseline errors, including:

- BluePeak Retail: PERSON → ORG
- Meridian Support: PERSON → ORG
- Apex Mobility: GPE → ORG
- Helena Costa: ORG → PERSON
- Porto: PERSON → GPE
- Valencia: PERSON → GPE
- Felix Bauer: ORG → PERSON
- Atlas Digital: PERSON → ORG

The custom pipeline also recovered organisations that were previously missed, including Orion Systems and SilverLine Distribution.

However, entity recognition alone still does not solve the complete information extraction task. Some temporal expressions remain incorrectly labelled, secondary dates and times may be extracted alongside primary event information, and operational fields such as event type, action and quantity require additional schema-level logic.

The next stage therefore separates entity detection from field selection and operational information extraction.

In [8]:
# Extract core structured fields from operational text
def extract_core_fields(record):
    text = record["text"]
    doc = nlp_custom(text)

    people = [
        ent.text for ent in doc.ents
        if ent.label_ == "PERSON"
    ]

    organisations = [
        ent.text for ent in doc.ents
        if ent.label_ == "ORG"
        and ent.text != "Operations"
    ]

    locations = [
        ent.text for ent in doc.ents
        if ent.label_ == "GPE"
    ]

    # Detect full calendar dates independently of spaCy
    date_pattern = (
        r"\b(?:[1-9]|[12]\d|3[01]) "
        r"(?:January|February|March|April|May|June|July|August|"
        r"September|October|November|December) "
        r"\d{4}\b"
    )

    dates = re.findall(date_pattern, text)

    # Detect clock times independently of spaCy
    # and exclude deadline expressions such as "by 11:00"
    # or "before 17:00"
    times = []

    for match in re.finditer(
        r"\b(?:[01]\d|2[0-3]):[0-5]\d\b",
        text
    ):
        preceding_text = text[
            max(0, match.start() - 12):match.start()
        ].lower()

        if re.search(r"\b(?:by|before)\s*$", preceding_text):
            continue

        times.append(match.group())

    # For rescheduled events, select the new date
    if "moved from" in text.lower():
        selected_date = dates[-1] if dates else None
    else:
        selected_date = dates[0] if dates else None

    selected_time = times[0] if times else None

    return {
        "id": record["id"],
        "person": people[0] if people else None,
        "organisation": organisations[0] if organisations else None,
        "location": locations[0] if locations else None,
        "date": selected_date,
        "time": selected_time
    }


core_records = [
    extract_core_fields(record)
    for record in corpus
]

core_df = pd.DataFrame(core_records)

core_df

,id,person,organisation,location,date,time
0,OP001,Maria Chen,Northstar Logistics,Madrid,4 August 2026,09:30
1,OP002,Daniel Foster,BluePeak Retail,Barcelona,6 August 2026,None
2,OP003,Sofia Mendes,Meridian Support,Lisbon,7 August 2026,None
3,OP004,Liam Murphy,Orion Systems,Dublin,8 August 2026,14:15
4,OP005,Elena Rossi,Apex Mobility,Milan,12 August 2026,15:00
5,OP006,James Walker,NovaCare Services,Warsaw,11 August 2026,None
6,OP007,Helena Costa,None,Porto,13 August 2026,22:00
7,OP008,None,SilverLine Distribution,Valencia,14 August 2026,None
8,OP009,Thomas Reed,BrightWave Telecom,London,15 August 2026,None
9,OP010,Camille Laurent,Horizon Connect,Paris,16 August 2026,10:45


In [9]:
# Extract operational event, action and quantity fields

event_rules = [
    ("warehouse system outage", "warehouse system outage"),
    ("transport disruption", "delivery delay"),
    ("staffing shortage", "staffing shortage"),
    ("payment platform outage", "payment platform outage"),
    ("service review", "service review rescheduled"),
    ("unresolved priority cases", "unresolved priority cases"),
    ("network maintenance", "scheduled network maintenance"),
    ("broke down", "delivery vehicle breakdown"),
    ("customer escalation", "customer escalation"),
    ("high call volumes", "unusually high call volumes"),
    ("cancelled the planned field inspection", "field inspection cancellation"),
    ("database performance issue", "database performance issue")
]


def extract_operational_fields(text):
    text_lower = text.lower()

    # Event type
    event_type = None

    for trigger, label in event_rules:
        if trigger in text_lower:
            event_type = label
            break

    # Action
    action = None

    action_patterns = [
        r"(restart the affected servers and provide an update by 11:00)",
        r"(assign six additional agents before the evening shift)",
        r"(escalated the incident to the infrastructure team)",
        r"(moved from 10 August 2026 to 12 August 2026 at 15:00)",
        r"(deploy a specialist support team)",
        r"(requested a replacement vehicle to prevent further delays)",
        r"(requested additional staffing)",
        r"(schedule a new inspection date)",
        r"(investigate slow response times before 17:00)"
    ]

    for pattern in action_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)

        if match:
            action = match.group(1)
            break

    # Quantity
    quantity = None

    quantity_patterns = [
        r"\b\d+\s+customer orders\b",
        r"\bsix additional agents\b",
        r"\b\d+\s+unresolved priority cases\b",
        r"\b\d+\s+accounts\b"
    ]

    for pattern in quantity_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)

        if match:
            quantity = match.group(0)
            break

    return {
        "event_type": event_type,
        "action": action,
        "quantity": quantity
    }


operational_records = []

for record in corpus:
    extracted = extract_operational_fields(record["text"])

    operational_records.append({
        "id": record["id"],
        **extracted
    })

operational_df = pd.DataFrame(operational_records)

operational_df

,id,event_type,action,quantity
0,OP001,warehouse system outage,restart the affected servers and provide an up...,None
1,OP002,delivery delay,None,240 customer orders
2,OP003,staffing shortage,assign six additional agents before the evenin...,six additional agents
3,OP004,payment platform outage,escalated the incident to the infrastructure team,None
4,OP005,service review rescheduled,moved from 10 August 2026 to 12 August 2026 at...,None
5,OP006,unresolved priority cases,deploy a specialist support team,38 unresolved priority cases
6,OP007,scheduled network maintenance,None,None
7,OP008,delivery vehicle breakdown,requested a replacement vehicle to prevent fur...,None
8,OP009,customer escalation,None,125 accounts
9,OP010,unusually high call volumes,requested additional staffing,None


In [10]:
# Combine all extracted fields into complete structured records
final_records = []

for core_record, operational_record in zip(
    core_records,
    operational_records
):
    final_records.append({
        **core_record,
        "event_type": operational_record["event_type"],
        "action": operational_record["action"],
        "quantity": operational_record["quantity"]
    })

extracted_df = pd.DataFrame(final_records)

# Evaluate exact field-level agreement with the gold standard
evaluation_fields = [
    "person",
    "organisation",
    "location",
    "date",
    "time",
    "event_type",
    "action",
    "quantity"
]

evaluation_results = []

for field in evaluation_fields:
    predicted = extracted_df[field].fillna("<MISSING>").astype(str)
    expected = gold_df[field].fillna("<MISSING>").astype(str)

    correct = predicted.eq(expected)

    evaluation_results.append({
        "Field": field,
        "Correct": int(correct.sum()),
        "Total": len(correct),
        "Accuracy": correct.mean()
    })

evaluation_df = pd.DataFrame(evaluation_results)

# Record-level exact match
record_matches = []

for i in range(len(extracted_df)):
    predicted_row = (
        extracted_df.loc[i, evaluation_fields]
        .fillna("<MISSING>")
        .astype(str)
    )

    expected_row = (
        gold_df.loc[i, evaluation_fields]
        .fillna("<MISSING>")
        .astype(str)
    )

    record_matches.append(
        predicted_row.equals(expected_row)
    )

overall_field_accuracy = (
    evaluation_df["Correct"].sum()
    / evaluation_df["Total"].sum()
)

print("Field-level evaluation:")
display(evaluation_df)

print(
    "\nOverall field accuracy:",
    f"{overall_field_accuracy:.2%}"
)

print(
    "Fully correct records:",
    f"{sum(record_matches)}/{len(record_matches)}"
)

Field-level evaluation:


,Field,Correct,Total,Accuracy
0,person,12,12,1.0
1,organisation,12,12,1.0
2,location,12,12,1.0
3,date,12,12,1.0
4,time,12,12,1.0
5,event_type,12,12,1.0
6,action,12,12,1.0
7,quantity,12,12,1.0



Overall field accuracy: 100.00%
Fully correct records: 12/12


## Controlled Evaluation

The complete extraction pipeline achieved exact agreement with the manually defined reference annotations on the 12-text development corpus.

Results:

- **96/96 individual fields correctly extracted**
- **100.00% field-level accuracy**
- **12/12 records fully correct**

This result confirms that the combined NER, rule-based and schema-selection logic correctly implements the target information structure for the controlled corpus.

However, these texts were also used during pipeline development and rule refinement. The 100% result should therefore not be interpreted as evidence of perfect generalisation to unseen operational language.

A separate robustness evaluation on previously unseen texts is performed next to measure how well the extraction logic transfers beyond the development examples.

In [11]:
# Previously unseen texts for robustness evaluation
robustness_corpus = [
    {
        "id": "RT001",
        "text": "Priya Nair from CedarPoint Freight reported that the inventory application went offline in Amsterdam on 20 August 2026 at 08:40. The support team will restore service and send a status report by 10:00."
    },
    {
        "id": "RT002",
        "text": "Marco Silva asked DeltaWorks Support to add four agents to the Rome evening shift on 21 August 2026 after unexpected absenteeism reduced coverage."
    },
    {
        "id": "RT003",
        "text": "Vertex Payments told Aisha Khan that card transactions in Brussels were failing intermittently at 13:20 on 22 August 2026. Engineers are investigating the gateway."
    },
    {
        "id": "RT004",
        "text": "The quarterly review with Daniel Ortiz and Lumina Mobility was postponed to 25 August 2026 at 16:30 in Vienna."
    },
    {
        "id": "RT005",
        "text": "A refrigerated van operated by Alpine Foods stopped working outside Lyon on 26 August 2026, delaying 57 deliveries. Dispatch arranged another vehicle."
    },
    {
        "id": "RT006",
        "text": "Nora Jensen from CloudBridge Networks escalated a high-priority connectivity complaint in Copenhagen on 27 August 2026 after outages affected 82 business customers."
    }
]

# Reference annotations fixed before running the existing extractor
robustness_gold = [
    {
        "id": "RT001",
        "person": "Priya Nair",
        "organisation": "CedarPoint Freight",
        "location": "Amsterdam",
        "date": "20 August 2026",
        "time": "08:40",
        "event_type": "inventory application outage",
        "action": "restore service and send a status report by 10:00",
        "quantity": None
    },
    {
        "id": "RT002",
        "person": "Marco Silva",
        "organisation": "DeltaWorks Support",
        "location": "Rome",
        "date": "21 August 2026",
        "time": None,
        "event_type": "staffing shortage",
        "action": "add four agents to the Rome evening shift",
        "quantity": "four agents"
    },
    {
        "id": "RT003",
        "person": "Aisha Khan",
        "organisation": "Vertex Payments",
        "location": "Brussels",
        "date": "22 August 2026",
        "time": "13:20",
        "event_type": "payment transaction failure",
        "action": "investigating the gateway",
        "quantity": None
    },
    {
        "id": "RT004",
        "person": "Daniel Ortiz",
        "organisation": "Lumina Mobility",
        "location": "Vienna",
        "date": "25 August 2026",
        "time": "16:30",
        "event_type": "service review rescheduled",
        "action": "postponed to 25 August 2026 at 16:30",
        "quantity": None
    },
    {
        "id": "RT005",
        "person": None,
        "organisation": "Alpine Foods",
        "location": "Lyon",
        "date": "26 August 2026",
        "time": None,
        "event_type": "delivery vehicle breakdown",
        "action": "arranged another vehicle",
        "quantity": "57 deliveries"
    },
    {
        "id": "RT006",
        "person": "Nora Jensen",
        "organisation": "CloudBridge Networks",
        "location": "Copenhagen",
        "date": "27 August 2026",
        "time": None,
        "event_type": "customer escalation",
        "action": None,
        "quantity": "82 business customers"
    }
]

robustness_gold_df = pd.DataFrame(robustness_gold)

print("Unseen robustness texts:", len(robustness_corpus))
print("Reference records:", len(robustness_gold_df))

robustness_gold_df

Unseen robustness texts: 6
Reference records: 6


,id,person,organisation,location,date,time,event_type,action,quantity
0,RT001,Priya Nair,CedarPoint Freight,Amsterdam,20 August 2026,08:40,inventory application outage,restore service and send a status report by 10:00,None
1,RT002,Marco Silva,DeltaWorks Support,Rome,21 August 2026,None,staffing shortage,add four agents to the Rome evening shift,four agents
2,RT003,Aisha Khan,Vertex Payments,Brussels,22 August 2026,13:20,payment transaction failure,investigating the gateway,None
3,RT004,Daniel Ortiz,Lumina Mobility,Vienna,25 August 2026,16:30,service review rescheduled,postponed to 25 August 2026 at 16:30,None
4,RT005,None,Alpine Foods,Lyon,26 August 2026,None,delivery vehicle breakdown,arranged another vehicle,57 deliveries
5,RT006,Nora Jensen,CloudBridge Networks,Copenhagen,27 August 2026,None,customer escalation,None,82 business customers


In [12]:
# Apply the existing pipeline to unseen robustness texts
robustness_core_records = [
    extract_core_fields(record)
    for record in robustness_corpus
]

robustness_operational_records = []

for record in robustness_corpus:
    extracted = extract_operational_fields(record["text"])

    robustness_operational_records.append({
        "id": record["id"],
        **extracted
    })

robustness_extracted_records = []

for core_record, operational_record in zip(
    robustness_core_records,
    robustness_operational_records
):
    robustness_extracted_records.append({
        **core_record,
        "event_type": operational_record["event_type"],
        "action": operational_record["action"],
        "quantity": operational_record["quantity"]
    })

robustness_extracted_df = pd.DataFrame(
    robustness_extracted_records
)

print("Extracted records from unseen texts:")
display(robustness_extracted_df)

# Evaluate against the unseen gold standard
robustness_evaluation = []

for field in evaluation_fields:
    predicted = (
        robustness_extracted_df[field]
        .fillna("<MISSING>")
        .astype(str)
    )

    expected = (
        robustness_gold_df[field]
        .fillna("<MISSING>")
        .astype(str)
    )

    correct = predicted.eq(expected)

    robustness_evaluation.append({
        "Field": field,
        "Correct": int(correct.sum()),
        "Total": len(correct),
        "Accuracy": correct.mean()
    })

robustness_evaluation_df = pd.DataFrame(
    robustness_evaluation
)

robustness_record_matches = []

for i in range(len(robustness_extracted_df)):
    predicted_row = (
        robustness_extracted_df.loc[i, evaluation_fields]
        .fillna("<MISSING>")
        .astype(str)
    )

    expected_row = (
        robustness_gold_df.loc[i, evaluation_fields]
        .fillna("<MISSING>")
        .astype(str)
    )

    robustness_record_matches.append(
        predicted_row.equals(expected_row)
    )

robustness_field_accuracy = (
    robustness_evaluation_df["Correct"].sum()
    / robustness_evaluation_df["Total"].sum()
)

print("\nField-level robustness evaluation:")
display(robustness_evaluation_df)

print(
    "\nOverall field accuracy:",
    f"{robustness_field_accuracy:.2%}"
)

print(
    "Fully correct records:",
    f"{sum(robustness_record_matches)}/{len(robustness_record_matches)}"
)

Extracted records from unseen texts:


,id,person,organisation,location,date,time,event_type,action,quantity
0,RT001,Priya Nair,CedarPoint Freight,Amsterdam,20 August 2026,08:40,None,None,None
1,RT002,Marco Silva,None,Rome,21 August 2026,None,None,None,None
2,RT003,Aisha Khan,Vertex Payments,Brussels,22 August 2026,13:20,None,None,None
3,RT004,Daniel Ortiz,None,Vienna,25 August 2026,16:30,None,None,None
4,RT005,Lyon,Alpine Foods,None,26 August 2026,None,None,None,None
5,RT006,Nora Jensen,CloudBridge Networks,None,27 August 2026,None,None,None,None



Field-level robustness evaluation:


,Field,Correct,Total,Accuracy
0,person,5,6,0.833333
1,organisation,4,6,0.666667
2,location,4,6,0.666667
3,date,6,6,1.000000
4,time,6,6,1.000000
5,event_type,0,6,0.000000
6,action,1,6,0.166667
7,quantity,3,6,0.500000



Overall field accuracy: 60.42%
Fully correct records: 0/6


## Robustness Evaluation on Unseen Text

The initial extraction pipeline was evaluated on six previously unseen operational texts without modifying any extraction rules.

Performance decreased substantially compared with the controlled development corpus:

- **60.42% overall field-level accuracy**
- **0/6 fully correct records**

Field-level performance was:

| Field | Accuracy |
|---|---:|
| Person | 83.33% |
| Organisation | 66.67% |
| Location | 66.67% |
| Date | 100.00% |
| Time | 100.00% |
| Event type | 0.00% |
| Action | 16.67% |
| Quantity | 50.00% |

The results reveal a clear difference between generalisable and overly specific extraction logic.

Date and time extraction transferred successfully to unseen wording, while pretrained NER continued to identify many new people, organisations and locations.

In contrast, event type and action extraction relied heavily on exact lexical triggers from the development corpus and failed to generalise to alternative formulations.

Additional NER errors also appeared in previously unseen entities, including missing organisations and locations and one location incorrectly classified as a person.

This evaluation demonstrates why perfect performance on a development corpus should not be interpreted as evidence of generalisation. The robustness set exposes where the pipeline requires more flexible linguistic patterns rather than additional memorised phrases.

In [13]:
# Improved extraction logic based on robustness error analysis

number_words = (
    "one|two|three|four|five|six|seven|eight|nine|ten"
)


def extract_core_fields_v2(record):
    text = record["text"]
    doc = nlp_custom(text)

    people = [
        ent.text for ent in doc.ents
        if ent.label_ == "PERSON"
    ]

    organisations = [
        ent.text for ent in doc.ents
        if ent.label_ == "ORG"
        and ent.text != "Operations"
    ]

    locations = [
        ent.text for ent in doc.ents
        if ent.label_ == "GPE"
    ]

    # Generic location fallback for capitalised place names
    # following common operational prepositions.
    location_match = re.search(
        r"\b(?:in|near|outside)\s+"
        r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)"
        r"(?=\s+(?:on|at|after|before|were|was|is|are|,)|\b)",
        text
    )

    if not locations and location_match:
        locations = [location_match.group(1)]

    selected_location = locations[0] if locations else None

    # Prevent a recovered location from also being retained
    # as a mistakenly classified person.
    if selected_location in people:
        people.remove(selected_location)

    # Generic organisation fallbacks for common sentence structures.
    if not organisations:
        organisation_patterns = [
            r"\bfrom\s+([A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+){1,3})"
            r"(?=\s+(?:reported|raised|escalated|confirmed))",

            r"\basked\s+([A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+){1,3})"
            r"\s+to\b",

            r"\band\s+([A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+){1,3})"
            r"\s+(?:was|were)\b",

            r"\boperated by\s+"
            r"([A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+){1,3})"
            r"(?=\s+(?:broke|stopped|failed))"
        ]

        for pattern in organisation_patterns:
            match = re.search(pattern, text)

            if match:
                organisations = [match.group(1)]
                break

    # Calendar dates independent of spaCy labels
    date_pattern = (
        r"\b(?:[1-9]|[12]\d|3[01]) "
        r"(?:January|February|March|April|May|June|July|August|"
        r"September|October|November|December) "
        r"\d{4}\b"
    )

    dates = re.findall(date_pattern, text)

    # Clock times independent of spaCy labels.
    # Deadline expressions such as "by 10:00" and
    # "before 17:00" are not treated as event times.
    times = []

    for match in re.finditer(
        r"\b(?:[01]\d|2[0-3]):[0-5]\d\b",
        text
    ):
        preceding_text = text[
            max(0, match.start() - 12):match.start()
        ].lower()

        if re.search(r"\b(?:by|before)\s*$", preceding_text):
            continue

        times.append(match.group())

    # Rescheduled events use the new date.
    if re.search(
        r"\b(?:moved|postponed|rescheduled)\b",
        text,
        flags=re.IGNORECASE
    ):
        selected_date = dates[-1] if dates else None
    else:
        selected_date = dates[0] if dates else None

    selected_time = times[0] if times else None

    return {
        "id": record["id"],
        "person": people[0] if people else None,
        "organisation": organisations[0] if organisations else None,
        "location": selected_location,
        "date": selected_date,
        "time": selected_time
    }


def extract_operational_fields_v2(text):
    text_lower = text.lower()

    # More flexible event categorisation
    event_type = None

    if "warehouse system" in text_lower and "outage" in text_lower:
        event_type = "warehouse system outage"

    elif (
        "inventory application" in text_lower
        and ("offline" in text_lower or "outage" in text_lower)
    ):
        event_type = "inventory application outage"

    elif (
        "payment platform" in text_lower
        and "outage" in text_lower
    ):
        event_type = "payment platform outage"

    elif (
        "card transactions" in text_lower
        and (
            "failing" in text_lower
            or "failed" in text_lower
            or "failure" in text_lower
        )
    ):
        event_type = "payment transaction failure"

    elif (
        "staffing shortage" in text_lower
        or "reduced coverage" in text_lower
        or "unexpected absenteeism" in text_lower
    ):
        event_type = "staffing shortage"

    elif (
        "review" in text_lower
        and re.search(
            r"\b(?:moved|postponed|rescheduled)\b",
            text_lower
        )
    ):
        event_type = "service review rescheduled"

    elif "unresolved priority cases" in text_lower:
        event_type = "unresolved priority cases"

    elif "network maintenance" in text_lower:
        event_type = "scheduled network maintenance"

    elif (
        re.search(r"\b(?:vehicle|van)\b", text_lower)
        and re.search(
            r"\b(?:broke down|stopped working|breakdown)\b",
            text_lower
        )
    ):
        event_type = "delivery vehicle breakdown"

    elif (
        "customer escalation" in text_lower
        or (
            "escalated" in text_lower
            and "complaint" in text_lower
        )
    ):
        event_type = "customer escalation"

    elif "high call volumes" in text_lower:
        event_type = "unusually high call volumes"

    elif (
        "field inspection" in text_lower
        and re.search(
            r"\b(?:cancelled|canceled)\b",
            text_lower
        )
    ):
        event_type = "field inspection cancellation"

    elif "database performance issue" in text_lower:
        event_type = "database performance issue"

    elif "transport disruption" in text_lower:
        event_type = "delivery delay"

    # Flexible follow-up action extraction
    action = None

    action_patterns = [
        r"\bwill\s+((?!begin\b|be delayed\b).+?)(?=\.)",

        r"\basked\s+[A-Z][A-Za-z]+"
        r"(?:\s+[A-Z][A-Za-z]+){1,3}\s+to\s+(.+?)(?=\.)",

        r"\brequested\s+(.+?)(?=\.)",

        r"\b(escalated the incident to .+?)(?=\.)",

        r"\b(?:engineers|the engineers)\s+are\s+(.+?)(?=\.)",

        r"\bwas\s+(postponed to .+?)(?=\.)",

        r"\b(arranged another vehicle)(?=\.)"
    ]

    for pattern in action_patterns:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            action = match.group(1)
            break

    # Generic operational quantity extraction
    quantity_pattern = (
        rf"\b(?:\d+|{number_words})\s+"
        r"(?:[A-Za-z-]+\s+){0,2}"
        r"(?:orders|agents|cases|accounts|deliveries|customers)\b"
    )

    quantity_match = re.search(
        quantity_pattern,
        text,
        flags=re.IGNORECASE
    )

    quantity = (
        quantity_match.group(0)
        if quantity_match
        else None
    )

    return {
        "event_type": event_type,
        "action": action,
        "quantity": quantity
    }


print("Improved extraction functions created.")

Improved extraction functions created.


In [14]:
# Re-evaluate the improved extractor on the robustness set
robustness_core_v2 = [
    extract_core_fields_v2(record)
    for record in robustness_corpus
]

robustness_operational_v2 = []

for record in robustness_corpus:
    extracted = extract_operational_fields_v2(record["text"])

    robustness_operational_v2.append({
        "id": record["id"],
        **extracted
    })

robustness_records_v2 = []

for core_record, operational_record in zip(
    robustness_core_v2,
    robustness_operational_v2
):
    robustness_records_v2.append({
        **core_record,
        "event_type": operational_record["event_type"],
        "action": operational_record["action"],
        "quantity": operational_record["quantity"]
    })

robustness_extracted_v2_df = pd.DataFrame(
    robustness_records_v2
)

print("Improved extraction on robustness texts:")
display(robustness_extracted_v2_df)

# Field-level evaluation
robustness_evaluation_v2 = []

for field in evaluation_fields:
    predicted = (
        robustness_extracted_v2_df[field]
        .fillna("<MISSING>")
        .astype(str)
    )

    expected = (
        robustness_gold_df[field]
        .fillna("<MISSING>")
        .astype(str)
    )

    correct = predicted.eq(expected)

    robustness_evaluation_v2.append({
        "Field": field,
        "Correct": int(correct.sum()),
        "Total": len(correct),
        "Accuracy": correct.mean()
    })

robustness_evaluation_v2_df = pd.DataFrame(
    robustness_evaluation_v2
)

record_matches_v2 = []

for i in range(len(robustness_extracted_v2_df)):
    predicted_row = (
        robustness_extracted_v2_df.loc[i, evaluation_fields]
        .fillna("<MISSING>")
        .astype(str)
    )

    expected_row = (
        robustness_gold_df.loc[i, evaluation_fields]
        .fillna("<MISSING>")
        .astype(str)
    )

    record_matches_v2.append(
        predicted_row.equals(expected_row)
    )

robustness_accuracy_v2 = (
    robustness_evaluation_v2_df["Correct"].sum()
    / robustness_evaluation_v2_df["Total"].sum()
)

print("\nImproved field-level evaluation:")
display(robustness_evaluation_v2_df)

print(
    "\nOverall field accuracy:",
    f"{robustness_accuracy_v2:.2%}"
)

print(
    "Fully correct records:",
    f"{sum(record_matches_v2)}/{len(record_matches_v2)}"
)

Improved extraction on robustness texts:


,id,person,organisation,location,date,time,event_type,action,quantity
0,RT001,Priya Nair,CedarPoint Freight,Amsterdam,20 August 2026,08:40,inventory application outage,restore service and send a status report by 10:00,None
1,RT002,Marco Silva,DeltaWorks Support,Rome,21 August 2026,None,staffing shortage,add four agents to the Rome evening shift on 2...,four agents
2,RT003,Aisha Khan,Vertex Payments,Brussels,22 August 2026,13:20,payment transaction failure,investigating the gateway,None
3,RT004,Daniel Ortiz,Lumina Mobility,Vienna,25 August 2026,16:30,service review rescheduled,postponed to 25 August 2026 at 16:30 in Vienna,None
4,RT005,None,Alpine Foods,Lyon,26 August 2026,None,delivery vehicle breakdown,arranged another vehicle,57 deliveries
5,RT006,Nora Jensen,CloudBridge Networks,Copenhagen,27 August 2026,None,customer escalation,None,82 business customers



Improved field-level evaluation:


,Field,Correct,Total,Accuracy
0,person,6,6,1.000000
1,organisation,6,6,1.000000
2,location,6,6,1.000000
3,date,6,6,1.000000
4,time,6,6,1.000000
5,event_type,6,6,1.000000
6,action,4,6,0.666667
7,quantity,6,6,1.000000



Overall field accuracy: 95.83%
Fully correct records: 4/6


## Error-Driven Pipeline Improvement

The robustness errors were analysed by failure type rather than corrected through record-specific memorisation.

The improved pipeline introduced more flexible logic for:

- organisation recovery from common sentence structures,
- location recovery from operational prepositions,
- correction of location/person conflicts,
- semantic event categorisation using alternative linguistic triggers,
- broader action extraction patterns,
- and generic operational quantity detection.

When re-evaluated on the six robustness texts, performance improved from **60.42% to 95.83% field-level accuracy**.

Seven of the eight target fields reached 100% accuracy:

- Person
- Organisation
- Location
- Date
- Time
- Event type
- Quantity

Action extraction reached **66.67% accuracy**.

The remaining action errors were primarily boundary errors rather than failures to identify the underlying action. In two cases, the correct action was extracted together with additional contextual information.

Overall, **4 of 6 records were completely correct**.

Because this robustness set informed pipeline improvement, it is no longer treated as an independent final evaluation set. The extraction logic is frozen at this point and will next be evaluated on a separate set of previously unseen texts.

In [15]:
# Final unseen holdout set
# The extraction pipeline is frozen before these texts are evaluated.

final_holdout_corpus = [
    {
        "id": "HT001",
        "text": "Owen Brooks from RedStone Logistics reported that the warehouse system was unavailable in Prague on 28 August 2026 at 07:50. Technicians will restart the platform and confirm recovery by 09:00."
    },
    {
        "id": "HT002",
        "text": "Unexpected absenteeism reduced coverage at the Stockholm service centre on 29 August 2026. Ingrid Larsen asked Polar Support to add five agents to the late shift."
    },
    {
        "id": "HT003",
        "text": "Summit Pay informed Yasmin Cole that card transactions in Zurich failed repeatedly at 12:35 on 30 August 2026. Engineers are checking the payment gateway."
    },
    {
        "id": "HT004",
        "text": "The monthly service review with Maya Singh and Aurora Mobility was rescheduled to 2 September 2026 at 14:30 in Budapest."
    },
    {
        "id": "HT005",
        "text": "A refrigerated van operated by Harbor Foods stopped working outside Marseille on 3 September 2026, delaying 64 deliveries. Dispatch arranged another vehicle."
    },
    {
        "id": "HT006",
        "text": "Leila Haddad from NovaLink Networks escalated a connectivity complaint in Oslo on 4 September 2026 after network outages affected 91 customers."
    }
]

# Reference annotations fixed before running the frozen pipeline
final_holdout_gold = [
    {
        "id": "HT001",
        "person": "Owen Brooks",
        "organisation": "RedStone Logistics",
        "location": "Prague",
        "date": "28 August 2026",
        "time": "07:50",
        "event_type": "warehouse system outage",
        "action": "restart the platform and confirm recovery by 09:00",
        "quantity": None
    },
    {
        "id": "HT002",
        "person": "Ingrid Larsen",
        "organisation": "Polar Support",
        "location": "Stockholm",
        "date": "29 August 2026",
        "time": None,
        "event_type": "staffing shortage",
        "action": "add five agents to the late shift",
        "quantity": "five agents"
    },
    {
        "id": "HT003",
        "person": "Yasmin Cole",
        "organisation": "Summit Pay",
        "location": "Zurich",
        "date": "30 August 2026",
        "time": "12:35",
        "event_type": "payment transaction failure",
        "action": "checking the payment gateway",
        "quantity": None
    },
    {
        "id": "HT004",
        "person": "Maya Singh",
        "organisation": "Aurora Mobility",
        "location": "Budapest",
        "date": "2 September 2026",
        "time": "14:30",
        "event_type": "service review rescheduled",
        "action": "rescheduled to 2 September 2026 at 14:30",
        "quantity": None
    },
    {
        "id": "HT005",
        "person": None,
        "organisation": "Harbor Foods",
        "location": "Marseille",
        "date": "3 September 2026",
        "time": None,
        "event_type": "delivery vehicle breakdown",
        "action": "arranged another vehicle",
        "quantity": "64 deliveries"
    },
    {
        "id": "HT006",
        "person": "Leila Haddad",
        "organisation": "NovaLink Networks",
        "location": "Oslo",
        "date": "4 September 2026",
        "time": None,
        "event_type": "customer escalation",
        "action": None,
        "quantity": "91 customers"
    }
]

final_holdout_gold_df = pd.DataFrame(
    final_holdout_gold
)

print(
    "Final unseen holdout texts:",
    len(final_holdout_corpus)
)

print(
    "Reference records:",
    len(final_holdout_gold_df)
)

final_holdout_gold_df

Final unseen holdout texts: 6
Reference records: 6


,id,person,organisation,location,date,time,event_type,action,quantity
0,HT001,Owen Brooks,RedStone Logistics,Prague,28 August 2026,07:50,warehouse system outage,restart the platform and confirm recovery by 0...,None
1,HT002,Ingrid Larsen,Polar Support,Stockholm,29 August 2026,None,staffing shortage,add five agents to the late shift,five agents
2,HT003,Yasmin Cole,Summit Pay,Zurich,30 August 2026,12:35,payment transaction failure,checking the payment gateway,None
3,HT004,Maya Singh,Aurora Mobility,Budapest,2 September 2026,14:30,service review rescheduled,rescheduled to 2 September 2026 at 14:30,None
4,HT005,None,Harbor Foods,Marseille,3 September 2026,None,delivery vehicle breakdown,arranged another vehicle,64 deliveries
5,HT006,Leila Haddad,NovaLink Networks,Oslo,4 September 2026,None,customer escalation,None,91 customers


In [16]:
# Final evaluation on the completely unseen holdout set
# The V2 extraction pipeline remains frozen.

final_core_records = [
    extract_core_fields_v2(record)
    for record in final_holdout_corpus
]

final_operational_records = []

for record in final_holdout_corpus:
    extracted = extract_operational_fields_v2(record["text"])

    final_operational_records.append({
        "id": record["id"],
        **extracted
    })

final_extracted_records = []

for core_record, operational_record in zip(
    final_core_records,
    final_operational_records
):
    final_extracted_records.append({
        **core_record,
        "event_type": operational_record["event_type"],
        "action": operational_record["action"],
        "quantity": operational_record["quantity"]
    })

final_extracted_df = pd.DataFrame(
    final_extracted_records
)

print("Final holdout extraction:")
display(final_extracted_df)

# Field-level evaluation
final_evaluation = []

for field in evaluation_fields:
    predicted = (
        final_extracted_df[field]
        .fillna("<MISSING>")
        .astype(str)
    )

    expected = (
        final_holdout_gold_df[field]
        .fillna("<MISSING>")
        .astype(str)
    )

    correct = predicted.eq(expected)

    final_evaluation.append({
        "Field": field,
        "Correct": int(correct.sum()),
        "Total": len(correct),
        "Accuracy": correct.mean()
    })

final_evaluation_df = pd.DataFrame(
    final_evaluation
)

# Fully correct records
final_record_matches = []

for i in range(len(final_extracted_df)):
    predicted_row = (
        final_extracted_df.loc[i, evaluation_fields]
        .fillna("<MISSING>")
        .astype(str)
    )

    expected_row = (
        final_holdout_gold_df.loc[i, evaluation_fields]
        .fillna("<MISSING>")
        .astype(str)
    )

    final_record_matches.append(
        predicted_row.equals(expected_row)
    )

final_field_accuracy = (
    final_evaluation_df["Correct"].sum()
    / final_evaluation_df["Total"].sum()
)

print("\nFinal field-level evaluation:")
display(final_evaluation_df)

print(
    "\nFinal overall field accuracy:",
    f"{final_field_accuracy:.2%}"
)

print(
    "Fully correct holdout records:",
    f"{sum(final_record_matches)}/{len(final_record_matches)}"
)

Final holdout extraction:


,id,person,organisation,location,date,time,event_type,action,quantity
0,HT001,Owen Brooks,RedStone Logistics,Prague,28 August 2026,07:50,None,restart the platform and confirm recovery by 0...,None
1,HT002,Ingrid Larsen,Polar Support,Stockholm,29 August 2026,None,staffing shortage,add five agents to the late shift,five agents
2,HT003,Yasmin Cole,None,Zurich,30 August 2026,12:35,payment transaction failure,checking the payment gateway,None
3,HT004,Maya Singh,Aurora Mobility,Budapest,2 September 2026,14:30,service review rescheduled,None,None
4,HT005,None,Harbor Foods,Marseille,3 September 2026,None,delivery vehicle breakdown,arranged another vehicle,64 deliveries
5,HT006,Leila Haddad,NovaLink Networks,Oslo,4 September 2026,None,customer escalation,None,91 customers



Final field-level evaluation:


,Field,Correct,Total,Accuracy
0,person,6,6,1.000000
1,organisation,5,6,0.833333
2,location,6,6,1.000000
3,date,6,6,1.000000
4,time,6,6,1.000000
5,event_type,5,6,0.833333
6,action,5,6,0.833333
7,quantity,6,6,1.000000



Final overall field accuracy: 93.75%
Fully correct holdout records: 3/6


## Final Holdout Evaluation

The frozen V2 extraction pipeline was evaluated on a final set of six previously unseen operational texts.

No extraction rules were modified after reviewing the holdout results.

The final evaluation achieved:

- **45/48 fields correctly extracted**
- **93.75% overall field-level accuracy**
- **3/6 records completely correct**

Field-level performance was:

| Field | Accuracy |
|---|---:|
| Person | 100.00% |
| Organisation | 83.33% |
| Location | 100.00% |
| Date | 100.00% |
| Time | 100.00% |
| Event type | 83.33% |
| Action | 83.33% |
| Quantity | 100.00% |

Five of the eight target fields achieved perfect accuracy on the final holdout set: person, location, date, time and quantity all transferred successfully to unseen wording.

Three remaining errors were observed:

- a new formulation of a warehouse system outage was not mapped to the existing event category,
- one previously unseen organisation was not recognised,
- and one rescheduling action was not extracted despite the event itself being correctly identified.

These results demonstrate that the error-driven improvements generalised substantially beyond the development examples while still revealing realistic limitations in rule-based NLP extraction.

The final holdout result is retained without post-evaluation rule modification to preserve the independence of the test set.

## Structured JSON Output

The final stage converts extracted operational information into a machine-readable JSON structure.

This format makes the NLP output suitable for downstream uses such as:

- workflow automation,
- incident management systems,
- operational dashboards,
- databases,
- APIs,
- quality monitoring,
- and human review workflows.

Missing information is preserved explicitly as `null` rather than inferred.

In [17]:
# Convert final holdout extraction to structured JSON
final_json_records = (
    final_extracted_df
    .where(pd.notnull(final_extracted_df), None)
    .to_dict(orient="records")
)

print(
    json.dumps(
        final_json_records,
        indent=2,
        ensure_ascii=False
    )
)

[
  {
    "id": "HT001",
    "person": "Owen Brooks",
    "organisation": "RedStone Logistics",
    "location": "Prague",
    "date": "28 August 2026",
    "time": "07:50",
    "event_type": null,
    "action": "restart the platform and confirm recovery by 09:00",
    "quantity": null
  },
  {
    "id": "HT002",
    "person": "Ingrid Larsen",
    "organisation": "Polar Support",
    "location": "Stockholm",
    "date": "29 August 2026",
    "time": null,
    "event_type": "staffing shortage",
    "action": "add five agents to the late shift",
    "quantity": "five agents"
  },
  {
    "id": "HT003",
    "person": "Yasmin Cole",
    "organisation": null,
    "location": "Zurich",
    "date": "30 August 2026",
    "time": "12:35",
    "event_type": "payment transaction failure",
    "action": "checking the payment gateway",
    "quantity": null
  },
  {
    "id": "HT004",
    "person": "Maya Singh",
    "organisation": "Aurora Mobility",
    "location": "Budapest",
    "date": "2 Septemb

## Conclusion

This project developed a hybrid NLP information extraction pipeline that transforms unstructured operational text into structured, machine-readable records.

The approach combined:

- pretrained spaCy Named Entity Recognition,
- domain-specific entity rules,
- regular-expression based date and time extraction,
- schema-level field selection,
- operational event categorisation,
- action extraction,
- quantity detection,
- error analysis,
- and structured JSON output.

A generic pretrained NER model provided a useful starting point but produced several entity classification errors and could not directly extract domain-specific operational fields.

On the controlled development corpus, the customised pipeline achieved **100% field-level accuracy**, but evaluation on unseen language exposed substantial overfitting in the initial rule design, with performance falling to **60.42%**.

Error analysis was then used to redesign the extraction logic around more general linguistic patterns rather than record-specific phrases.

The improved pipeline reached **95.83% field-level accuracy** on the error-analysis set.

Most importantly, the frozen pipeline was evaluated on a separate final holdout set that had not been used for rule development.

Final holdout performance was:

- **93.75% overall field-level accuracy**
- **45 of 48 fields correctly extracted**
- **3 of 6 records completely correct**
- **100% accuracy for person, location, date, time and quantity**
- **83.33% accuracy for organisation, event type and action**

The remaining errors demonstrate realistic limitations of rule-based NLP systems when previously unseen entities or linguistic formulations appear.

The project therefore illustrates not only how structured information can be extracted from operational text, but also why independent evaluation, error analysis and explicit treatment of missing information are essential when building reliable NLP pipelines.

The final output is represented as JSON, making the extracted information suitable for downstream systems such as operational dashboards, workflow automation, incident management, databases, APIs and human review processes.